# شام — مسار جمع وتدريب أداة ترميز الصورة (VQ-VAE) — CPU، بلا حاجة لـ GPU

نظير دفتر الصوت تماماً، لكن للصورة. **نطاق حقيقي يجب معرفته بصراحة:** تعليقات الصور (captions) العربية الأصلية شحيحة — لكن هذا لا يهم في هذا الدفتر تحديداً: تدريب أداة ترميز الصورة نفسها (VQ-VAE) **لا يحتاج تعليقات إطلاقاً** (تعلّم غير مُشرَف بالكامل، إعادة بناء بكسلات فقط) — التعليقات تُستخدم لاحقاً فقط في دفتر التدريب متعدد الوسائط (فهم/توليد الصورة من نص). لذلك هذا الدفتر يجمع صوراً عربية حقيقية (مع تعليقاتها إن وُجدت، تُحفظ للاستخدام لاحقاً) لكنه يتدرّب على البكسلات وحدها.

يحاول أولاً مصدراً عربياً حقيقياً (`Misraj/Arabic-Image-Captioning_100M`)، ويتراجع تلقائياً لـ Flickr30k (تعليقات إنجليزية، لكن الصور نفسها حقيقية ومتنوعة وهذا كل ما يلزم لهذه المرحلة) إن فشل المصدر العربي — تحقّق حقيقي وقت التشغيل وليس افتراضاً، تماماً كما في دفتر المرحلة الثانية.

كل تشغيل: يجمع دفعة **جديدة** من الصور الحقيقية لم تُستخدم سابقاً، يكمل تدريب نفس أداة الترميز، وينشرها كمجموعة بيانات Kaggle خاصة بهذا المسار — جاهزة ليستخدمها لاحقاً دفتر التدريب متعدد الوسائط.

## قبل "Save Version → Save & Run All":
1. **فعّل الإنترنت** من Settings (لا حاجة لـ GPU).
2. تأكد من وجود نفس أسرار Kaggle: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY`.
3. **من التشغيل الثاني فصاعداً**: أضف نتاج هذا الدفتر نفسه كـ Input ليكمل التدريب.
4. استخدم **Save Version → Save & Run All (Commit)** دائماً، ويمكن جدولته للتشغيل التلقائي اليومي.


### 1) سحب الكود الحقيقي من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("كود شام الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
try:
    import datasets
    print(f"مكتبة datasets متوفرة مسبقاً (نسخة {datasets.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "datasets"], check=True)
    print("تم تثبيت مكتبة datasets.")


### 3) استئناف أداة ترميز الصورة من التشغيل السابق (إن وُجد)

نفس المبدأ تماماً — حجم **إنتاجي كامل** (256×256، 8192 رمزاً، القيم الافتراضية في `image_tokenizer.py`) وليس نسخة مصغّرة، لأن هذا المسار مخصص لبنائها تدريجياً عبر تشغيلات غير محدودة.

In [ ]:
import json as _json
from pathlib import Path
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig
from tokenizer_select import select_pretrained_tokenizer, _progress_samples

# الاستئناف: يُفحص كل ما في المدخلات ويُختار الأكثر تدريباً؛ النسخ غير المتوافقة
# (قاموس بحجم مختلف، أو محفوظة بإصدار أقدم) تُصلَح وتُقبل ولا تُتجاهل (tokenizer_select.py).
samples_consumed = 0
start_step = 0
print("البحث عن نسخة سابقة من أداة الترميز (image):")
_picked = select_pretrained_tokenizer("image")
if _picked:
    image_tokenizer, start_step, _resume_path = _picked
    image_tokenizer_cfg = image_tokenizer.cfg
    samples_consumed = _progress_samples(_resume_path, "image")
    print(f"استؤنف من: {_resume_path} (خطوة {start_step:,}، {samples_consumed:,} عيّنة سابقاً)")
else:
    image_tokenizer_cfg = ImageTokenizerConfig()
    image_tokenizer = ImageTokenizer(image_tokenizer_cfg)
    print("لا توجد نسخة سابقة صالحة — بدء أداة ترميز جديدة (متوقَّع فقط في أول تشغيل).")


### 4) جمع دفعة جديدة من صور حقيقية (عربية أولاً، Flickr30k عند الفشل)

`skip=samples_consumed` يضمن رؤية صور **جديدة** كل تشغيل.

In [ ]:
from sham_data_sources import Ledger, collect

MAX_IMAGES = 1_000

# جمع بلا تكرار: مصادر موثّقة متعددة بالترتيب (عند نفاد أحدها أو تعطّله يُنتقل للتالي
# تلقائياً)، وموضع استئناف حقيقي لكل مصدر، وبصمة لكل عيّنة سبق جمعها — تُدمج من كل
# المدخلات، فلا يعيد أي دفتر جمع ما جمعه هو أو دفتر آخر من قبل (sham_data_sources.py).
image_ledger = Ledger.load("image")
image_manifest, collect_stats = collect("image", "/kaggle/working/corpus/images", MAX_IMAGES, image_ledger, image_size=image_tokenizer_cfg.image_size)
NEW_DATA = collect_stats.written > 0
if not NEW_DATA:
    print("⚠ لا توجد بيانات جديدة غير مكررة — لن يُعاد التدريب على بيانات سبق استخدامها. "
          "إن ظهر أن كل المصادر استُنفدت فالمسار أنهى مهمته، أو أضف مصدراً جديداً إلى SOURCES.")
print(f"جاهز: {image_manifest}")


### 5) قياس سرعة CPU الحقيقية ثم التدريب ضمن ميزانية زمنية محددة

In [ ]:
if NEW_DATA:
    import time
    import torch
    from PIL import Image
    from train_image_tokenizer import train_vqvae

    image_root = Path(image_manifest).parent
    images_list = []
    with open(image_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            img = Image.open(image_root / record["image"]).convert("RGB").resize((image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size))
            tensor = torch.tensor(list(img.getdata()), dtype=torch.float32).view(image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size, 3)
            images_list.append(tensor.permute(2, 0, 1) / 127.5 - 1.0)
    real_images = torch.stack(images_list, dim=0)
    print(f"عدد الصور الحقيقية المحمَّلة: {real_images.shape[0]:,}")

    BATCH_SIZE = 8
    MAX_TRAINING_MINUTES = 45

    t0 = time.time()
    _ = train_vqvae(image_tokenizer, real_images[: min(BATCH_SIZE, real_images.shape[0])], num_epochs=1, batch_size=BATCH_SIZE, log_every=999)
    seconds_per_epoch_calib = (time.time() - t0) * (real_images.shape[0] / min(BATCH_SIZE, real_images.shape[0]))
    seconds_per_epoch = max(seconds_per_epoch_calib, 0.01)
    num_epochs = max(3, min(60, int((MAX_TRAINING_MINUTES * 60 * 0.85) / seconds_per_epoch)))
    print(f"سرعة حقيقية مقاسة: ~{seconds_per_epoch:.2f} ثانية/حقبة على كامل البيانات -> {num_epochs} حقبة ضمن {MAX_TRAINING_MINUTES} دقيقة.")

    stats = train_vqvae(image_tokenizer, real_images, num_epochs=num_epochs, batch_size=BATCH_SIZE, log_every=5)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {stats.epoch_losses[-1]:.4f} | استخدام القاموس: {stats.final_codebook_usage}/{stats.codebook_size}")

else:
    stats = None
    print('تخطّي التدريب: لا بيانات جديدة.')


### 6) حفظ ونشر أداة ترميز الصورة (كمجموعة بيانات Kaggle خاصة بهذا المسار)

In [ ]:
from train_image_tokenizer import save_tokenizer_checkpoint

trained_epochs = len(stats.epoch_losses) if NEW_DATA else 0
final_step = start_step + trained_epochs
final_samples_consumed = samples_consumed + (collect_stats.written if NEW_DATA else 0)

ckpt_dir = Path("/kaggle/working/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
save_tokenizer_checkpoint(ckpt_dir / "image_tokenizer.pt", image_tokenizer, step=final_step)
(ckpt_dir / "image_tokenizer_progress.json").write_text(
    _json.dumps({"samples_consumed": final_samples_consumed, "step": final_step, "last_run_sources": collect_stats.per_source})
)
image_ledger.save(ckpt_dir)  # مواضع الاستئناف + بصمات كل ما جُمع — يرافق نقطة الحفظ دائماً
print(f"تم الحفظ عند الخطوة {final_step:,} (إجمالي العيّنات: {final_samples_consumed:,}) | {collect_stats.summary()}")


In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-image-tokenizer-checkpoint"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
_shutil.copy2("/kaggle/working/checkpoints/image_tokenizer.pt", upload_dir / "image_tokenizer.pt")
_shutil.copy2("/kaggle/working/checkpoints/image_tokenizer_progress.json", upload_dir / "image_tokenizer_progress.json")
_shutil.copy2("/kaggle/working/checkpoints/image_ledger.json", upload_dir / "image_ledger.json")

metadata = {"title": "sham-image-tokenizer-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# نفس الفحص الحتمي المعتمد في كل الدفاتر الأخرى: نسأل Kaggle مباشرة هل
# مجموعة البيانات هذه موجودة أصلاً بدل تخمين ذلك من نص رسالة خطأ (غير
# موثّق وقد يتغيّر) — هذا الأسلوب هو ما كشف خطأ "-r skip" الصامت سابقاً.
_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

# "-r zip" وليس "skip": القيمة الافتراضية الموثّقة لـ Kaggle CLI للمجلدات
# الفرعية هي تجاهلها بصمت، لا رفعها — درس مستفاد من خطأ حقيقي سابق أثّر
# على كل دفاتر التدريب النصية قبل اكتشافه.
if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "تم النشر إلى" if _dataset_exists else "تم إنشاء"
    print(f"{verb} {DATASET_SLUG} — التشغيل المجدول القادم سيلتقطها تلقائياً ويكمل من حيث توقفنا.")
elif "incompatible" in _combined.lower():
    print(
        "تحذير: هذه المجموعة أُنشئت سابقاً بصيغة رفع غير متوافقة. لا يوجد إصلاح على مستوى الكود لها تحديداً — "
        "غيّر الاسم أعلاه لاسم لم يُستخدم من قبل (مثلاً أضف -v2) وأعد التشغيل. نقطة الحفظ نفسها آمنة "
        "في Output هذه الجلسة بغض النظر."
    )
    print(_combined)
else:
    print("تحذير: فشل النشر — نقطة الحفظ لا تزال آمنة في Output هذه الجلسة. تأكد من صحة KAGGLE_USERNAME/KAGGLE_KEY.")
    print(_combined)
